# Solution E2 · The production model does not match the validated one

**Type**: Find the bug

## Objective

Find out why the probabilities computed with `model.get_booster().predict(DMatrix)` (used "in production", for explanations and for export) differ from `model.predict_proba` (used in validation) and why the per-example contributions **do not sum** to the margin of the scikit-learn wrapper: after early stopping the booster contains
**all** the trained trees, including the 30 added after the minimum, and the wrapper ignores them automatically while the native booster does not.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
booster = model.get_booster()
dte = xgb.DMatrix(X_te)

## Commented solution

In [ ]:
validated_p = model.predict_proba(X_te)[:, 1]                        # in validation
production_p = booster.predict(dte)                                  # 'in production we use the booster: it is faster'
max_diff = float(np.abs(production_p - validated_p).max())
validated_ll, production_ll = log_loss(y_te, validated_p), log_loss(y_te, production_p)
print(f"maximum gap between the probabilities: {max_diff:.4f} | validated log-loss {validated_ll:.4f}, in production {production_ll:.4f}")

**Where the bug is.** `fit` with early stopping **does not remove** the trees after the minimum: the booster contains `best_iteration + 1` good trees plus the 30 of "patience". `model.predict_proba` automatically uses only the first `best_iteration + 1`; `booster.predict` (the native API) uses all of them, as does `pred_contribs`. You need
`iteration_range=(0, best_iteration + 1)`.

In [ ]:
best = int(model.best_iteration)
total_trees = booster.num_boosted_rounds()
extra_trees = total_trees - (best + 1)
print(f"best_iteration = {best} -> good trees {best + 1}, trees in the booster {total_trees} ({extra_trees} more)")

In [ ]:
ok_p = booster.predict(dte, iteration_range=(0, best + 1))
ok_p_matches = bool(np.allclose(ok_p, validated_p, atol=1e-6))
contribs = booster.predict(dte, pred_contribs=True, iteration_range=(0, best + 1))
margin = model.predict(X_te, output_margin=True)
sum_ok = bool(np.allclose(contribs.sum(1), margin, atol=1e-4))
print("the booster with iteration_range reproduces predict_proba:", ok_p_matches, "| the contributions sum to the margin of the wrapper:", sum_ok)

In [ ]:
bug_contribs = booster.predict(dte, pred_contribs=True)                       # without iteration_range: all the trees
bug_sum_gap = float(np.abs(bug_contribs.sum(1) - margin).max())
top_bug = np.abs(bug_contribs[:, :-1]).mean(0).argmax()
top_ok = np.abs(contribs[:, :-1]).mean(0).argmax()
top_changes = bool(top_bug != top_ok)
print(f"maximum gap between the sum of the contributions (without iteration_range) and the margin: {bug_sum_gap:.3f} | the most important feature changes: {top_changes} (bug: f{top_bug}, correct: f{top_ok})")

## Check

In [ ]:
with verify():
    check(max_diff > 0.03 and production_ll > validated_ll, "the colleague sees a gap of over 0.03 between the probabilities and a worse log-loss in production", "don't modify the colleague's code")
    check(best is not None and total_trees is not None and total_trees == best + 31 and extra_trees == 30, "the booster has best_iteration + 31 trees: the 30 rounds of patience are inside the model", "booster.num_boosted_rounds()")
    check(ok_p is not None and ok_p_matches is True, "with iteration_range=(0, best + 1) the booster reproduces predict_proba", "iteration_range=(0, best_iteration + 1)")
    check(contribs is not None and contribs.shape == (len(X_te), 13) and sum_ok is True, "13 columns (12 features and the bias) and sum equal to the margin of the wrapper", "pred_contribs=True with the same iteration_range")
    check(bug_sum_gap is not None and bug_sum_gap > 0.3 and top_changes in (True, False), "without iteration_range the sum of the contributions departs by over 0.3 from the validated margin", "sum of the contributions minus margin")
done()

## What you learned

- **`best_iteration` is an attribute, not a truncation:** the booster always contains all the trained trees (here 126, against the 96 good ones). `XGBClassifier.predict` and `predict_proba` know how to use `best_iteration`, the native API (`Booster.predict`, `pred_contribs`) does not unless you specify `iteration_range`.
- **Effect on the probabilities:** maximum gap of 0.11 between the two versions and log-loss 0.385 against 0.380 on the test set: a small but systematic difference, which in production shows up as "the model got worse compared with the validation" without anyone understanding why.
- **Effect on the explanations:** the sum of the contributions computed with all the trees departs by 0.86 from the validated margin: the explanation concerns a different model from the one that predicts. The mean importance per feature often does not change (the top features stay the same), but the decomposition of the single case is no longer reliable.
- **Practical rule:** either always pass `iteration_range=(0, best_iteration + 1)` to the native APIs, or retrain with `n_estimators = best_iteration + 1` without early stopping before exporting. A useful regression test: `np.allclose(booster.predict(...), wrapper.predict_proba(...))` before going to production.

## Common pitfall

Extracting the booster (`get_booster()`), saving it or using it for SHAP/contributions believing it contains only the "best" model: it also contains the patience rounds. Check `num_boosted_rounds()` and pass `iteration_range`.

## Interview question

<details><summary>After a fit with early stopping, does the booster contain the trees of the best round?</summary>

No, it contains all the trained trees (best round plus the patience rounds). The scikit-learn wrapper uses only the first best_iteration + 1 in predict, but the native API (Booster.predict, pred_contribs, SHAP) uses everything unless you pass iteration_range=(0, best_iteration + 1). Before exporting I compare the predictions of the booster with those
of the wrapper, or I retrain with the optimal number of rounds.

</details>